### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.

This notebook will:
1. Delete all resources and purge soft-deleted services
2. Delete the resource group
3. Sweep for any remaining soft-deleted Azure ML workspaces that might block redeployment

In [ ]:
import os, sys
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group = f"lab-{deployment_name}"

utils.cleanup_resources(deployment_name, resource_group_name=resource_group)

### 🧹 Purge any remaining soft-deleted ML workspaces

If the resource group was already deleted, soft-deleted Azure ML workspaces may still linger and block redeployment with the same names. This cell sweeps for and purges them.

In [ ]:
dep_list = utils.run(f'az deployment group list -g {resource_group} --query "[].name" -o json', "", "", print_output=False)
if dep_list.success and dep_list.json_data:
    for dep_name in dep_list.json_data:
        utils.run(f'az deployment group delete -g {resource_group} -n {dep_name}', "", "", print_output=False)
    utils.print_info(f"Cleared {len(dep_list.json_data)} stale deployment record(s)")

sub_output = utils.run('az account show --query id -o tsv', "", "", print_output=False)
if sub_output.success:
    sub_id = sub_output.text.strip()
    ml_output = utils.run(
        f'az rest --method GET --url "/subscriptions/{sub_id}/resourceGroups/{resource_group}/providers/Microsoft.MachineLearningServices/workspaces?api-version=2024-04-01&includeDeletedWorkSpaces=true" --query "value[?properties.provisioningState==\'SoftDeleted\'].name" -o json',
        "", "", print_output=False
    )
    if ml_output.success and ml_output.json_data:
        for name in ml_output.json_data:
            utils.print_info(f"Purging soft-deleted ML Workspace: {name}")
            utils.run(
                f'az rest --method DELETE --url "/subscriptions/{sub_id}/resourceGroups/{resource_group}/providers/Microsoft.MachineLearningServices/workspaces/{name}?api-version=2024-04-01&forceToPurge=true"',
                f"Purged {name}", f"Failed to purge {name}"
            )

utils.print_ok("Soft-deleted resource purge complete")